# ThinkGuard — StudyChat 1,087건을 두 그룹으로 처리

기존 `thinkguard_team_last3000.zip`과 원 작업자의 `full/run_config.json`을 사용합니다. **StudyChat 1,087건만 처리하고 Recipe4U 1,913건은 제외합니다.** 학생 발화·이전 대화·원본 ID와 해시는 바꾸거나 자르지 않습니다.

- 일반 그룹: 실제 입력 **30,000토큰 이하**. 대표 8건으로 배치 2·4·8의 처리 시간을 비교하고 선택합니다.
- 긴 그룹: **30,000토큰 초과**. 한 건씩 처리합니다.
- 엔진은 최대 길이 **69,632토큰**으로 한 번만 로드합니다. 상한은 입력을 그 길이로 늘리는 설정이 아닙니다. 긴 그룹의 최장 입력을 먼저 시험합니다.
- 토큰 길이·배치 선택·결과를 Drive에 저장해 재개할 때 재사용합니다. 실패 결과는 성공으로 세지 않습니다.

**처음 전환할 때:** 기존 7,168토큰 엔진이 이미 올라가 있다면 **런타임 → 세션 다시 시작**을 한 번 하고 위에서부터 실행하세요. 이후 같은 엔진으로 셀을 재실행할 때는 모델을 재사용합니다. 패키지 버전이 맞으면 설치를 건너뜁니다.

결과는 새 `qwen35_9b_vllm_team_studychat1087_split_v2` 폴더에 저장합니다. 기존 결과는 유지합니다. 모델·리비전·프롬프트·출력 길이는 원 작업자와 대조하되, 승인한 엔진 길이·배치·입력 한도 차이는 그룹별 설정에 기록합니다. **원 작업자의 기존 단일 설정 병합 함수로 바로 합칠 수 없으며**, 전달 ZIP에는 그룹별 `run_config.json`과 `run_manifest.json`을 포함합니다.


## 1. L4 GPU 확인

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("런타임 → 런타임 유형 변경 → GPU로 설정한 뒤 다시 실행하세요.")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name)
print("VRAM GiB:", round(gpu.total_memory / 2**30, 2))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
if gpu.total_memory / 2**30 < 14:
    raise RuntimeError("이 노트북의 9B 시험 설정에는 14 GiB 이상 GPU를 권장합니다. 설정을 먼저 점검하세요.")


## 2. Drive 연결, 팀원 ZIP 및 원 작업자 설정 업로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ruff: noqa: F821
import hashlib
import json
import shutil
import sys
import zipfile
from pathlib import Path

# Google Drive는 위에서 이미 mount했다고 가정
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks")

RUN_NAME = "qwen35_9b_vllm_team_studychat1087_split_v2"
NORMAL_INPUT_CUTOFF = 30000
ENGINE_MAX_MODEL_LEN = 69632
LONG_MAX_INPUT_TOKENS = 68608
NORMAL_BATCH_CANDIDATES = (2, 4, 8)
LONG_BATCH_SIZE = 1
TARGET_DATASET = "studychat"
EXPECTED_COUNT = 1087
BUNDLE_NAME = "thinkguard_team_last3000.zip"

# Google Drive에 저장된 ZIP 파일
BUNDLE = PROJECT_DIR / BUNDLE_NAME

# Colab 로컬 작업 디렉터리
WORK = Path("/content/thinkguard_team_studychat1087")
WORK.mkdir(parents=True, exist_ok=True)

# Drive에 ZIP 파일이 있는지 확인
if not BUNDLE.exists():
    raise FileNotFoundError(f"ZIP 파일을 찾을 수 없습니다: {BUNDLE}")

# Drive의 ZIP 파일을 Colab 작업 디렉터리로 복사
shutil.copyfile(BUNDLE, WORK / "bundle.zip")

# ZIP 검증 및 압축 해제
with zipfile.ZipFile(WORK / "bundle.zip") as archive:
    if archive.testzip() is not None:
        raise ValueError("ZIP checksum error")

    for name in archive.namelist():
        if Path(name).name != name or name.startswith("."):
            raise ValueError("Unexpected archive path")

    archive.extractall(WORK)

# Manifest 검증
manifest = json.loads(
    (WORK / "manifest.json").read_text(encoding="utf-8")
)

for name, expected in manifest["files"].items():
    with (WORK / name).open("rb") as stream:
        observed = hashlib.file_digest(stream, "sha256").hexdigest()

    if observed != expected:
        raise ValueError(f"업로드 파일 검증 실패: {name}")

sys.path.insert(0, str(WORK))

print("원본 행 수:", manifest["counts"])
print("Drive 저장 위치:", PROJECT_DIR)


# 원본 실행 설정 검증
from team_split import assert_compatible

owner_config_path = PROJECT_DIR / "run_config.json"

if not owner_config_path.exists():
    raise FileNotFoundError(
        f"원래 full/run_config.json 파일을 찾을 수 없습니다: "
        f"{owner_config_path}"
    )

owner_config = json.loads(
    owner_config_path.read_text(encoding="utf-8")
)

assignment = json.loads(
    (WORK / "assignment.json").read_text(encoding="utf-8")
)

if owner_config["bundle_inputs_sha256"] != assignment["parent_inputs_sha256"]:
    raise ValueError("다른 원본의 실행 설정입니다.")

if owner_config["model"].get("engine") != "vllm":
    raise ValueError("vLLM 원본 실행 설정이 필요합니다.")

if owner_config["model"]["model_id"] != "RedHatAI/Qwen3.5-9B-FP8-dynamic":
    raise ValueError("예상한 FP8 모델이 아닙니다.")

from runner import read_inputs
from journal import record_key, stable_hash

# 순서나 앞쪽 행 수에 의존하지 않고 데이터셋 이름으로 선택합니다.
studychat_inputs = [
    row for row in read_inputs(WORK / "inputs.jsonl.gz")
    if row["dataset"] == TARGET_DATASET
]
studychat_keys = [record_key(row) for row in studychat_inputs]
assigned_studychat_keys = [
    key for key in assignment["team_keys"]
    if key.startswith(TARGET_DATASET + ":")
]
if manifest["counts"].get(TARGET_DATASET) != EXPECTED_COUNT or len(studychat_inputs) != EXPECTED_COUNT:
    raise ValueError(f"StudyChat 건수가 예상과 다릅니다: {len(studychat_inputs)} / {EXPECTED_COUNT}")
if len(set(studychat_keys)) != EXPECTED_COUNT or studychat_keys != assigned_studychat_keys:
    raise ValueError("StudyChat 원본 ID 또는 배정 순서가 일치하지 않습니다.")

print(f"실행 대상: StudyChat {len(studychat_inputs):,}건")
print(f"추론 제외: Recipe4U {manifest['counts'].get('recipe4u', 0):,}건")
print("원본 배정 정보는 유지하며, StudyChat 결과만 별도 폴더에 저장합니다.")

## 3. 패키지 버전 확인 — 이미 맞으면 설치 생략

원 작업자의 고정 버전과 다를 때만 설치합니다. 실제 설치로 패키지 버전이 바뀐 경우에만 세션 재시작이 필요합니다. 안내가 나온 상태에서 설치 셀만 반복 실행해도 재시작을 대신하지 않습니다.


In [ ]:
# ruff: noqa: F821
import importlib.metadata
import subprocess
import sys
import os

packages = ["vllm", "torch", "transformers", "safetensors", "sentencepiece", "huggingface-hub"]
environment_file = PROJECT_DIR / "colab_vllm_package_versions.json"

def versions():
    result = {}
    for name in packages:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result

before = versions()
wanted = owner_config["model"]["versions"]
requirements = [
    f"{name}=={version}" for name, version in wanted.items()
    if before.get(name) != version
]
requirements += [name for name in ("safetensors", "sentencepiece") if before[name] is None]
if requirements:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *requirements])
else:
    print("필요한 버전이 이미 설치되어 있어 설치를 건너뜁니다.")

installed = versions()
if any(installed.get(name) != version for name, version in wanted.items()):
    raise RuntimeError("설치된 패키지가 원 작업자의 고정 버전과 다릅니다.")
if any(value is None for value in installed.values()):
    raise RuntimeError("필수 패키지가 설치되지 않았습니다.")
environment_file.write_text(json.dumps(installed, indent=2), encoding="utf-8")
print(installed)
restart_marker = WORK / "restart_required.json"
if before != installed:
    restart_marker.write_text(json.dumps({"pid": os.getpid()}), encoding="utf-8")
if restart_marker.exists():
    if json.loads(restart_marker.read_text())["pid"] == os.getpid():
        raise RuntimeError("패키지 버전이 변경되었습니다. 런타임 → 세션 다시 시작을 한 번 한 뒤 위에서부터 실행하세요.")
    restart_marker.unlink()
print("패키지 확인 완료. 다음 셀을 실행하세요.")


### 토큰 길이 확인·두 그룹 분리

첫 실행에서 길이를 측정해 저장하고, 같은 데이터·모델·프롬프트·토크나이저 버전으로 재개하면 저장한 길이를 사용합니다. 모든 발화와 이전 대화를 그대로 유지합니다.


In [ ]:
# ruff: noqa: F821
from contract import chat_messages, SYSTEM, PROMPT_VERSION
from journal import atomic_json, record_key, stable_hash

def round_up_tokens(value):
    return ((value + 1023) // 1024) * 1024

def partition_by_length(inputs, token_counts, cutoff):
    if len(inputs) != len(token_counts) or any(type(n) is not int or n <= 0 for n in token_counts):
        raise ValueError("입력과 토큰 길이 목록이 일치하지 않습니다.")
    keys = [record_key(row) for row in inputs]
    if len(keys) != len(set(keys)) or any(row["dataset"] != "studychat" for row in inputs):
        raise ValueError("StudyChat 외 데이터 또는 중복 ID가 있습니다.")
    lengths = dict(zip(keys, token_counts))
    groups = {"normal": [], "long": []}
    for source, count in zip(inputs, token_counts):
        groups["normal" if count <= cutoff else "long"].append(source)
    # 비슷한 길이끼리 묶되, 같은 대화의 입력은 가까이 둡니다.
    for sources in groups.values():
        sources.sort(key=lambda row: (
            lengths[record_key(row)] // 2048,
            row["conversation_id"],
            lengths[record_key(row)],
            record_key(row),
        ))
    if sum(len(sources) for sources in groups.values()) != len(inputs):
        raise ValueError("그룹 분리 중 입력이 누락되었습니다.")
    return groups, lengths

run_root = PROJECT_DIR / RUN_NAME
run_root.mkdir(exist_ok=True)
length_identity = stable_hash({
    "inputs_sha256": manifest["inputs_sha256"],
    "keys": studychat_keys,
    "model_id": owner_config["model"]["model_id"],
    "revision": owner_config["model"]["revision"],
    "prompt_sha256": stable_hash(SYSTEM),
    "transformers": installed["transformers"],
    "enable_thinking": False,
})
length_cache_path = run_root / "token_lengths.json"
cached_lengths = json.loads(length_cache_path.read_text(encoding="utf-8")) if length_cache_path.exists() else {}
if cached_lengths.get("identity") == length_identity:
    studychat_token_counts = cached_lengths["counts"]
    print("저장한 토큰 길이를 재사용합니다.")
else:
    from transformers import AutoTokenizer
    length_tokenizer = AutoTokenizer.from_pretrained(
        owner_config["model"]["model_id"],
        revision=owner_config["model"]["revision"],
        trust_remote_code=False,
    )
    studychat_token_counts = []
    for index, source in enumerate(studychat_inputs, 1):
        token_ids = length_tokenizer.apply_chat_template(
            chat_messages(source), add_generation_prompt=True,
            tokenize=True, return_dict=False, enable_thinking=False,
        )
        if not isinstance(token_ids, list) or not token_ids or any(type(t) is not int or t < 0 for t in token_ids):
            raise TypeError("토크나이저 결과가 정수 토큰 ID 목록이 아닙니다.")
        studychat_token_counts.append(len(token_ids))
        if index % 200 == 0:
            print(f"토큰 길이 확인: {index:,}/{EXPECTED_COUNT:,}건")
    del length_tokenizer

group_inputs, length_by_key = partition_by_length(
    studychat_inputs, studychat_token_counts, NORMAL_INPUT_CUTOFF,
)
atomic_json(length_cache_path, {"identity": length_identity, "counts": studychat_token_counts})
MAX_NEW_TOKENS = owner_config["max_new_tokens"]
retry_headroom = MAX_NEW_TOKENS + 1024
max_initial_tokens = max(studychat_token_counts)
if max_initial_tokens + retry_headroom > LONG_MAX_INPUT_TOKENS:
    raise RuntimeError("최장 입력에 필요한 재시도 여유가 설정 상한을 넘습니다. 원문을 자르지 않고 중단합니다.")
if LONG_MAX_INPUT_TOKENS + MAX_NEW_TOKENS > ENGINE_MAX_MODEL_LEN:
    raise RuntimeError("입력 한도와 출력 공간이 엔진 길이를 넘습니다.")
normal_max = max((length_by_key[record_key(row)] for row in group_inputs["normal"]), default=0)
group_input_limits = {
    "normal": min(LONG_MAX_INPUT_TOKENS, round_up_tokens(normal_max + retry_headroom)),
    "long": LONG_MAX_INPUT_TOKENS,
}
print("StudyChat 전체:", len(studychat_inputs), "건 / 최장 입력:", max_initial_tokens, "토큰")
for group, sources in group_inputs.items():
    print(f"{group}: {len(sources):,}건 / 재시도 포함 입력 한도: {group_input_limits[group]:,}")
print("엔진 최대 길이:", ENGINE_MAX_MODEL_LEN, "/ 출력 최대 길이:", MAX_NEW_TOKENS)


## 4. 두 그룹이 공유할 FP8 모델 로드

최대 길이 69,632토큰의 엔진을 한 번만 로드합니다. 긴 입력의 실제 처리 가능 여부는 다음 셀에서 확인합니다. 메모리 부족이 나면 모델 로드 셀을 반복 실행하지 말고 오류 내용을 확인하세요.


In [ ]:
# ruff: noqa: F821
import copy
import importlib.metadata
import os
from inference_vllm import Annotator
from runner import configuration
from journal import Journal, atomic_json, record_key, stable_hash
from team_split import assert_compatible

MODEL_ID = owner_config["model"]["model_id"]
ENGINE_OVERRIDES = {
    "max_model_len": ENGINE_MAX_MODEL_LEN,
    "max_num_seqs": max(NORMAL_BATCH_CANDIDATES),
    "max_num_batched_tokens": 8192,
    "enable_chunked_prefill": True,
}
engine_settings = copy.deepcopy(owner_config["model"]["engine_settings"])
engine_settings.update(ENGINE_OVERRIDES)

def load_split_model(model_id, revision, settings):
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
    import torch
    from vllm import LLM
    if not torch.cuda.is_available():
        raise RuntimeError("GPU 런타임이 필요합니다.")
    loaded = LLM(
        model=model_id, revision=revision, tokenizer_revision=revision,
        trust_remote_code=False, **settings,
    )
    metadata = {
        "model_id": model_id, "revision": revision, "engine": "vllm",
        "quantization": "checkpoint FP8 dynamic",
        "engine_settings": copy.deepcopy(settings),
        "versions": {name: importlib.metadata.version(name) for name in (
            "torch", "vllm", "transformers", "huggingface-hub",
        )},
        "gpu": torch.cuda.get_device_name(0),
        "vram_gib": torch.cuda.get_device_properties(0).total_memory / 2**30,
    }
    return loaded, loaded.get_tokenizer(), metadata

def make_group_config(group, batch_size):
    group_annotator = Annotator(
        model, tokenizer, max_input_tokens=group_input_limits[group],
        max_new_tokens=MAX_NEW_TOKENS,
    )
    group_annotator.batch_size = batch_size
    config = configuration(manifest, model_metadata, group_annotator)
    # 승인한 길이·배치 변경만 허용하고, 모델·리비전·버전·프롬프트는 계속 검증합니다.
    expected = copy.deepcopy(owner_config)
    expected["max_input_tokens"] = group_input_limits[group]
    expected["batch_size"] = batch_size
    expected["model"]["engine_settings"].update(ENGINE_OVERRIDES)
    assert_compatible(expected, config)
    config["input_selection"] = {
        "dataset": TARGET_DATASET, "group": group,
        "count": len(group_inputs[group]),
        "keys_sha256": stable_hash(sorted(record_key(row) for row in group_inputs[group])),
        "history_policy": "full_original", "normal_input_cutoff": NORMAL_INPUT_CUTOFF,
    }
    config["execution_profile"] = {
        "name": "studychat_two_groups_v2",
        "owner_config_sha256": stable_hash(owner_config),
        "owner_exactly_compatible": False,
        "approved_overrides": {
            "engine_settings": ENGINE_OVERRIDES,
            "max_input_tokens": group_input_limits[group],
            "batch_size": batch_size,
        },
    }
    return group_annotator, config

def validate_records(records, sources, config, require_complete=False):
    expected = {record_key(source): source for source in sources}
    if len(expected) != len(sources):
        raise ValueError("원본 입력 ID가 중복됩니다.")
    checked = {}
    values = records.values() if isinstance(records, dict) else records
    for record in values:
        key = record_key(record)
        if key in checked or key not in expected:
            raise ValueError("중복 또는 예상 밖 결과 ID: " + key)
        source = expected[key]
        if record.get("run_signature") != stable_hash(config):
            raise ValueError("결과와 실행 설정의 서명이 다릅니다: " + key)
        if any(record.get(field) != source[field] for field in (
            "source_sha256", "input_sha256", "student_id", "conversation_id",
        )):
            raise ValueError("결과와 원본 입력 정보가 다릅니다: " + key)
        if not isinstance(record.get("status"), str):
            raise ValueError("결과 상태가 없습니다: " + key)
        checked[key] = record
    if require_complete and set(checked) != set(expected):
        raise ValueError(f"결과가 누락되었습니다: {len(checked)}/{len(expected)}")
    return checked

def combine_attempts(attempts, sources, config, require_complete=False):
    combined = {}
    for round_index, records in enumerate(attempts):
        for key, record in validate_records(records, sources, config).items():
            # 이미 성공한 결과를 덮어쓰지 않고, 실패한 결과만 후속 시도의 결과로 대체합니다.
            if key not in combined or combined[key]["status"] != "ok":
                combined[key] = {**record, "execution_round": round_index}
    validate_records(combined, sources, config, require_complete)
    return combined

def merge_group_results(results, configs, sources, groups):
    merged = {}
    for group, group_sources in groups.items():
        checked = validate_records(results[group], group_sources, configs[group], True)
        for key, record in checked.items():
            if key in merged:
                raise ValueError("그룹 사이에 중복 ID가 있습니다: " + key)
            merged[key] = {**record, "execution_group": group}
    keys = [record_key(source) for source in sources]
    if len(keys) != len(set(keys)) or set(merged) != set(keys):
        raise ValueError("병합 결과가 전체 StudyChat 원본 ID와 일치하지 않습니다.")
    return [merged[key] for key in keys]

if (WORK / "restart_required.json").exists():
    raise RuntimeError("패키지 설치 셀의 재시작 안내를 먼저 해결하세요.")
if "model" in globals():
    if (model_metadata.get("model_id") != MODEL_ID
        or model_metadata.get("revision") != owner_config["model"]["revision"]
        or model_metadata.get("engine_settings") != engine_settings):
        raise RuntimeError("기존 모델의 엔진 길이 또는 설정이 다릅니다. 세션을 한 번 다시 시작하고 위에서부터 실행하세요.")
    print("이미 로드된 동일 설정의 모델을 재사용합니다.")
else:
    print("FP8 모델을 로드합니다. 엔진 길이:", ENGINE_MAX_MODEL_LEN)
    model, tokenizer, model_metadata = load_split_model(
        MODEL_ID, owner_config["model"]["revision"], engine_settings,
    )
atomic_json(run_root / "current_hardware.json", model_metadata)
print("GPU:", model_metadata["gpu"], "/ 엔진:", model_metadata["engine"])
print("모델·리비전·프롬프트는 유지하고, 두 그룹의 길이·배치 변경은 별도 설정에 기록합니다.")


## 5. 최장 입력 시험·일반 그룹 배치 선택

긴 그룹에서 가장 긴 1건을 먼저 시험합니다. 일반 그룹은 길이 분포를 대표하는 최대 8건으로 배치 2·4·8의 처리 시간을 비교합니다. 캐시를 비운 조건에서 주석 처리 시간(토큰화·생성·검증 포함)을 측정하며, Drive 저장 시간은 비교에서 제외합니다.

성공한 표본은 본 실행에 재사용합니다. 선택한 배치는 저장해 재개할 때 바꾸거나 재측정하지 않습니다. 이는 소량 표본에 따른 선택이며 전체 작업의 최단 시간을 보장하는 측정은 아닙니다.


In [ ]:
# ruff: noqa: F821
from runner import run, export_review

def representative_probe(sources, count=8):
    ordered = sorted(sources, key=lambda source: (length_by_key[record_key(source)], record_key(source)))
    count = min(count, len(ordered))
    if count <= 1:
        return ordered[:count]
    return [ordered[(len(ordered) - 1) * i // (count - 1)] for i in range(count)]

def check_pilot(journal, sources, config, description):
    records = validate_records(journal.records, sources, config, True)
    failures = {key: record["status"] for key, record in records.items() if record["status"] != "ok"}
    if failures:
        review = export_review(journal, sources)
        raise RuntimeError(f"{description} 표본 오류: {failures}. 검토 파일: {review}")
    return records

# 엔진이 가장 긴 실제 입력을 처리할 수 있는지 먼저 확인합니다.
long_annotator, long_config = make_group_config("long", LONG_BATCH_SIZE)
long_probe = representative_probe(group_inputs["long"], 1)
if group_inputs["long"]:
    long_probe = [max(group_inputs["long"], key=lambda source: length_by_key[record_key(source)])]
long_pilot = run(long_probe, long_annotator, run_root / "pilot/long_b1", long_config)
check_pilot(long_pilot, long_probe, long_config, "긴 그룹 최장 입력")

normal_probe = representative_probe(group_inputs["normal"])
selection_identity = stable_hash({
    "normal_probe": [record_key(row) for row in normal_probe],
    "configs": {str(batch): make_group_config("normal", batch)[1] for batch in NORMAL_BATCH_CANDIDATES},
})
selection_path = run_root / "batch_selection.json"
if selection_path.exists():
    batch_selection = json.loads(selection_path.read_text(encoding="utf-8"))
    if batch_selection.get("identity") != selection_identity:
        raise RuntimeError("저장한 배치 선택과 설정이 다릅니다. 새 RUN_NAME을 사용하세요.")
    normal_batch_size = batch_selection["selected_batch_size"]
    if normal_batch_size not in NORMAL_BATCH_CANDIDATES:
        raise ValueError("저장한 배치 크기가 후보 목록에 없습니다.")
    print("저장한 일반 그룹 배치를 재사용합니다:", normal_batch_size)
else:
    measurements = []
    for batch_size in NORMAL_BATCH_CANDIDATES:
        candidate, config = make_group_config("normal", batch_size)
        # 모든 generate 호출이 끝난 상태에서 후보 간 공유 문맥 캐시의 영향을 제거합니다.
        if not model.reset_prefix_cache():
            raise RuntimeError("배치 측정 전 캐시를 비우지 못했습니다. 진행 중인 요청을 확인하세요.")
        directory = run_root / f"pilot/normal_b{batch_size}"
        journal = run(normal_probe, candidate, directory, config)
        records = validate_records(journal.records, normal_probe, config, True)
        all_ok = all(record["status"] == "ok" for record in records.values())
        # 각 배치의 시간이 모든 행에 반복 기록되어 있으므로 배치 크기로 나눠 합산합니다.
        compute_seconds = sum(
            record["batch_seconds"] / record["batch_size"] for record in records.values()
        )
        measurements.append({
            "batch_size": batch_size, "seconds": compute_seconds,
            "all_ok": all_ok, "count": len(records),
        })
        print(f"일반 그룹 배치 {batch_size}: 표본 {len(records)}건 / {compute_seconds:.2f}초 / 정상={all_ok}")
    valid = [result for result in measurements if result["all_ok"]]
    if not valid:
        raise RuntimeError("모든 일반 그룹 배치 표본에 오류가 있습니다. pilot 결과를 확인하세요.")
    winner = min(valid, key=lambda result: result["seconds"])
    normal_batch_size = winner["batch_size"]
    batch_selection = {
        "identity": selection_identity, "selected_batch_size": normal_batch_size,
        "measurements": measurements, "hardware": model_metadata,
    }
    atomic_json(selection_path, batch_selection)

normal_annotator, normal_config = make_group_config("normal", normal_batch_size)
normal_pilot = Journal(run_root / f"pilot/normal_b{normal_batch_size}", normal_config)
check_pilot(normal_pilot, normal_probe, normal_config, "일반 그룹")
print("일반 그룹 선택 배치:", normal_batch_size, "/ 긴 그룹 배치:", LONG_BATCH_SIZE)
print("모든 표본이 정상입니다. 본 실행 셀로 진행하세요.")


## 6. 두 그룹 본 실행·재개

일반 그룹을 선택한 배치로 처리한 뒤 긴 그룹을 1건씩 처리합니다. 표본의 성공 결과는 복사해 다시 추론하지 않습니다. 저장된 ID는 건너뜁니다.

처리 완료 후 실패 건이 남으면 출력에서 원인을 확인하세요. **실패 건만 다시 시도하려면 `RETRY_ROUND`를 1로 바꾸고 이 셀과 결과 셀을 다시 실행**하세요. 추가 재시도는 2, 3처럼 늘립니다. 기존 성공 결과와 체크포인트는 유지합니다.


In [ ]:
# ruff: noqa: F821
RUN_FULL = True
RETRY_ROUND = 0  # 첫 실행은 0. 실패 건만 추가 시도하려면 1, 2, ... 로 올립니다.

def run_group(group, group_annotator, config, pilot):
    sources = group_inputs[group]
    directory = run_root / group / "full"
    saved = Journal(directory, config)
    validate_records(saved.records, sources, config)
    for record in check_pilot(pilot, long_probe if group == "long" else normal_probe, config, group).values():
        if record_key(record) not in saved.records:
            saved.add(record)
    saved.flush()
    base = run(sources, group_annotator, directory, config)
    attempts = [base.records]
    best = combine_attempts(attempts, sources, config, True)
    for round_index in range(1, RETRY_ROUND + 1):
        pending = [source for source in sources if best[record_key(source)]["status"] != "ok"]
        retry_directory = run_root / group / f"retry_{round_index:02d}"
        retry = run(pending, group_annotator, retry_directory, config)
        attempts.append(retry.records)
        best = combine_attempts(attempts, sources, config, True)
    statuses = {}
    for record in best.values():
        statuses[record["status"]] = statuses.get(record["status"], 0) + 1
    print(f"{group} 처리 결과:", statuses)
    return best

if type(RETRY_ROUND) is not int or RETRY_ROUND < 0:
    raise ValueError("RETRY_ROUND는 0 이상의 정수여야 합니다.")
if RUN_FULL:
    group_configs = {"normal": normal_config, "long": long_config}
    group_results = {
        "normal": run_group("normal", normal_annotator, normal_config, normal_pilot),
        "long": run_group("long", long_annotator, long_config, long_pilot),
    }
else:
    raise RuntimeError("표본 결과를 확인한 뒤 RUN_FULL=True로 실행하세요.")


## 7. 두 그룹 병합·완료 검사·ZIP 다운로드

그룹별 실행 설정·원본 해시·결과 ID를 검증하고 원본 StudyChat 순서로 합칩니다. **1,087건 모두 정상 상태일 때만 완료 ZIP을 다운로드**합니다. 실패가 남으면 실행 보고서에 ID와 상태를 기록하고 중단합니다.

이 ZIP은 원 작업자의 단일 실행 설정과 다릅니다. 기존 `team_split.merge_records`에 그대로 전달하지 말고, 함께 들어 있는 그룹별 설정과 보고서를 기준으로 병합해야 합니다.


In [ ]:
# ruff: noqa: F821
from google.colab import files
from collections import Counter
import os
import zipfile

combined = merge_group_results(
    group_results, group_configs, studychat_inputs, group_inputs,
)
if len(combined) != EXPECTED_COUNT:
    raise RuntimeError(f"StudyChat 결과 수가 다릅니다: {len(combined)} / {EXPECTED_COUNT}")
statuses = dict(Counter(record["status"] for record in combined))
failed = [{"id": record_key(record), "status": record["status"]} for record in combined if record["status"] != "ok"]
run_manifest = {
    "dataset": TARGET_DATASET, "expected": EXPECTED_COUNT, "processed": len(combined),
    "statuses": statuses, "failed": failed, "retry_round": RETRY_ROUND,
    "all_rows_ok": not failed,
    "normal_input_cutoff": NORMAL_INPUT_CUTOFF,
    "owner_exactly_compatible": False,
    "owner_config_sha256": stable_hash(owner_config),
    "original_assignment_sha256": manifest["files"]["assignment.json"],
    "groups": {
        group: {
            "count": len(group_inputs[group]),
            "run_signature": stable_hash(group_configs[group]),
            "config_in_zip": f"{group}/run_config.json",
        } for group in group_inputs
    },
}
annotation_path = run_root / "annotations.jsonl"
temporary_path = annotation_path.with_suffix(".tmp")
temporary_path.write_text(
    "".join(json.dumps(record, ensure_ascii=False) + "\n" for record in combined),
    encoding="utf-8",
)
os.replace(temporary_path, annotation_path)
atomic_json(run_root / "run_manifest.json", run_manifest)
print("StudyChat 병합 결과:", statuses)
if failed:
    raise RuntimeError(
        f"{len(failed)}건이 아직 정상 완료되지 않았습니다. run_manifest.json을 확인하고 "
        "본 실행 셀의 RETRY_ROUND를 한 단계 올려 실패 건만 재시도하세요."
    )
result_zip = PROJECT_DIR / "thinkguard_team_studychat1087_split_results.zip"
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(annotation_path, "annotations.jsonl")
    archive.write(run_root / "run_manifest.json", "run_manifest.json")
    archive.write(WORK / "assignment.json", "assignment.json")
    archive.write(selection_path, "batch_selection.json")
    for group in group_inputs:
        archive.write(run_root / group / "full/run_config.json", f"{group}/run_config.json")
print(f"StudyChat {EXPECTED_COUNT:,}건 정상 처리 완료. Recipe4U는 제외했습니다.")
print("전달 파일:", result_zip)
files.download(str(result_zip))
